# Diagnostico estrátegico integral para plataforma de suscripción y delivery

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio de aplicación de deliveys con suscrpción premium para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **catalog.csv** → costos de productos, categorías y proveedores  
- **marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python)

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

In [ ]:
# importar librerías
import pandas as pd

In [ ]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [ ]:
# explorar datasets
orders.head(5)

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [ ]:
catalog.head(5)

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [ ]:
marketing.head(5)

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


---

### Revisión y calidad de datos

In [ ]:
# Explorar información de datasets
print('========= ORDERS ==========')
orders.info()
print()
print ('========== CATALOGO ==========')
catalog.info()
print()
print('========== MARKETING ==========')
marketing.info()

========= ORDERS ==========
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB

========== CATALOGO ==========
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 col

In [ ]:
# Convertir fechas al formato correcto
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'], errors = 'coerce')
marketing['fecha'] = pd.to_datetime(marketing['fecha'], errors = 'coerce')

# Validar cambio
orders.info()
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25100 non-null  object        
 1   id_usuario          25100 non-null  object        
 2   fecha_hora_pedido   25100 non-null  datetime64[ns]
 3   pais                24800 non-null  object        
 4   dispositivo         25080 non-null  object        
 5   fuente_referencia   25070 non-null  object        
 6   nombre_producto     25070 non-null  object        
 7   categoria_producto  25020 non-null  object        
 8   cantidad            25050 non-null  float64       
 9   precio_unitario     25050 non-null  float64       
 10  monto_descuento     25050 non-null  float64       
 11  monto_total         25100 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.3+ MB
<class 'pandas.core.frame.DataFrame'

In [ ]:
# Revisar variables numéricas (sin negativos o ceros inválidos)
# Orders
# Revisar valores menores o iguales a 0 en cantidad
print('========== ORDERS ==========')
print('Registros con cantidad <= 0:')
print(orders['cantidad'].le(0).sum())

# Revisar valores negativos en precio_unitario
print('Registros con precio_unitario < 0:')
print(orders['precio_unitario'].lt(0).sum())

# Revisar montos en 0 o negativos
print('Registros con monto_total <= 0:')
print(orders['monto_total'].le(0).sum())

# Revisar valores negativos en monto_descuento
print('Registros con monto_descuento < 0:')
print(orders['monto_descuento'].lt(0).sum())
print()

# Revisar nulos en Orders
print('Valores nulos')
print(orders.isnull().sum())

# Catalog
# Revisar valores menores o iguales a 0 en costo_unitario
print('========== CATALOG ==========')
print('Registros con costo_unitario <= 0:')
print(catalog['costo_unitario'].le(0).sum())
print()

# Marketing
# Revisar valores negativos en  gasto
print('========== MARKETING ==========')
print('Registros con gasto < 0:')
print(marketing['gasto'].lt(0).sum())

========== ORDERS ==========
Registros con cantidad <= 0:
4
Registros con precio_unitario < 0:
0
Registros con monto_total <= 0:
4
Registros con monto_descuento < 0:
0

Valores nulos
id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     80
cantidad               50
precio_unitario        50
monto_descuento        50
monto_total             0
dtype: int64
========== CATALOG ==========
Registros con costo_unitario <= 0:
0

========== MARKETING ==========
Registros con gasto < 0:
0


In [ ]:
# Ver los registros problemáticos
print("Registros con cantidad <= 0:")
problemas_cantidad = orders[orders['cantidad'] <= 0]
print(problemas_cantidad[['id_pedido', 'cantidad', 'precio_unitario', 'monto_total']])

Registros con cantidad <= 0:
     id_pedido  cantidad  precio_unitario  monto_total
266  order_266      -2.0           101.31      -192.62
267  order_267      -1.0            43.50       -38.50
268  order_268      -1.0           497.65      -492.65
269  order_269      -1.0           423.53      -423.53


In [ ]:
# Ver registros con nulos en cualquier columna
registros_con_nulos = orders[orders.isnull().any(axis=1)]
print(f"Registros con nulos: {len(registros_con_nulos)}")
registros_con_nulos.head()

Registros con nulos: 400


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
24,order_24,user_458,2025-01-05,Colombia,NaN,organic,Sneakers-Urban-42,Moda,2.0,117.99,0.0,235.99
25,order_25,user_540,2025-03-08,Argentina,NaN,social,Vacuum-Pro-Black,Hogar,1.0,183.50,5.0,178.50
26,order_26,user_1951,2025-02-10,Colombia,NaN,organic,Blender-XL-Red,Hogar,1.0,331.52,15.0,316.52
27,order_27,user_5410,2025-05-09,colombia,NaN,paid_search,Phone-Pro-128GB,Electronica,2.0,449.48,5.0,893.96
28,order_28,user_1489,2025-01-17,Mexico,NaN,social,Phone-Pro-128GB,Electronica,1.0,60.70,10.0,50.70


In [ ]:
# Eliminar los 50 registros con nulos en variables críticas
orders_final = orders.dropna(subset=['cantidad', 'precio_unitario', 'monto_descuento'])

# Verificar el resultado
print(f"Registros antes: {len(orders)}")
print(f"Registros después: {len(orders_final)}")
print(f"Nulos restantes: {orders_final.isnull().sum().sum()}")

# Actualizar el dataset
orders = orders_final.copy()

Registros antes: 25100
Registros después: 25050
Nulos restantes: 410


In [ ]:
# Eliminar registros con cantidad  y monto <= 0
orders = orders[(orders['cantidad'] > 0) | (orders['cantidad'].isna())].copy()

# Validar cambio
print('========== ORDERS ==========')
print('Registros con cantidad <= 0:')
print(orders['cantidad'].le(0).sum())
print('Registros con monto_total <= 0:')
print(orders['monto_total'].le(0).sum())

========== ORDERS ==========
Registros con cantidad <= 0:
0
Registros con monto_total <= 0:
0


In [ ]:
# Verificar consistencia de montos
# Calcular monto_total esperado y verificar diferencias
orders['monto_total_calculado'] = (orders['cantidad'] * orders['precio_unitario'] - orders['monto_descuento'])
orders['diferencia'] = abs(orders['monto_total'] - orders['monto_total_calculado'])

# Contar registros con diferencias significativas (> 0.01)
inconsistentes = orders[orders['diferencia'] > 0.01]
print(f"Registros con inconsistencias: {len(inconsistentes)}")

# Ver algunos ejemplos si hay inconsistencias
if len(inconsistentes) > 0:
    print(inconsistentes[['id_pedido', 'cantidad', 'precio_unitario',
                         'monto_descuento', 'monto_total', 'monto_total_calculado',
                         'diferencia']].head())

Registros con inconsistencias: 1146
     id_pedido  cantidad  precio_unitario  monto_descuento  monto_total  \
2      order_2       2.0           102.99             10.0       195.99   
24    order_24       2.0           117.99              0.0       235.99   
35    order_35       2.0           467.34              5.0       929.69   
41    order_41       2.0            37.77             10.0        65.53   
136  order_136       2.0           211.05             15.0       407.09   

     monto_total_calculado  diferencia  
2                   195.98        0.01  
24                  235.98        0.01  
35                  929.68        0.01  
41                   65.54        0.01  
136                 407.10        0.01  


In [ ]:
# Eliminar columnas auxiliares
orders = orders.drop(['monto_total_calculado', 'diferencia'], axis=1)

In [ ]:
# Buscar duplicados orders
duplicados_id = orders.duplicated(subset=['id_pedido'], keep=False)

# Ver los registros que tienen id_pedido duplicado
orders[duplicados_id].sort_values('id_pedido')

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
25023,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.67
10082,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.67
25037,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10
10709,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10
25065,order_10829,user_7697,2025-01-23,Argentina,mobile,social,Phone-Pro-128GB,Electronica,2.0,115.54,0.0,231.08
...,...,...,...,...,...,...,...,...,...,...,...,...
25048,order_8326,user_6177,2025-06-14,Argentina,mobile,organic,Vacuum-Pro-Black,Hogar,1.0,457.53,0.0,457.53
25043,order_8414,user_4073,2025-05-10,Colombia,mobile,social,Jacket-Winter-M,Moda,2.0,144.35,10.0,278.71
8414,order_8414,user_4073,2025-05-10,Colombia,mobile,social,Jacket-Winter-M,Moda,2.0,144.35,10.0,278.71
25056,order_974,user_3262,2025-05-04,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,2.0,268.58,5.0,532.16


In [ ]:
# Eliminar duplicados por id_pedido, manteniendo el primero
orders_sin_duplicados = orders.drop_duplicates(subset=['id_pedido'], keep='first')

# Verificar el resultado
print(f"Registros originales: {len(orders)}")
print(f"Registros después de eliminar duplicados: {len(orders_sin_duplicados)}")
print(f"Duplicados eliminados: {len(orders) - len(orders_sin_duplicados)}")

Registros originales: 25046
Registros después de eliminar duplicados: 24946
Duplicados eliminados: 100


In [ ]:
# Actualizar el dataset
orders = orders_sin_duplicados.copy()
orders.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 24946 entries, 0 to 24999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           24946 non-null  object        
 1   id_usuario          24946 non-null  object        
 2   fecha_hora_pedido   24946 non-null  datetime64[ns]
 3   pais                24650 non-null  object        
 4   dispositivo         24926 non-null  object        
 5   fuente_referencia   24916 non-null  object        
 6   nombre_producto     24916 non-null  object        
 7   categoria_producto  24916 non-null  object        
 8   cantidad            24946 non-null  float64       
 9   precio_unitario     24946 non-null  float64       
 10  monto_descuento     24946 non-null  float64       
 11  monto_total         24946 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.5+ MB


In [ ]:
# Buscar duplicados en catalog por nombre de producto
print("Duplicados por nombre de producto:")
print(catalog.duplicated(subset=['nombre_producto']).sum())

# Buscar duplicados en marketing por combinación clave
print("Duplicados por fecha + país + canal:")
print(marketing.duplicated(subset=['fecha', 'pais', 'canal']).sum())

Duplicados por nombre de producto:
0
Duplicados por fecha + país + canal:
66


In [ ]:
# Ver algunos ejemplos de estos duplicados en Marketing
duplicados_marketing = marketing[marketing.duplicated(subset=['fecha', 'pais', 'canal'], keep=False)]
print(f"Registros con combinación duplicada: {len(duplicados_marketing)}")

# Ver ejemplos específicos
duplicados_marketing.sort_values(['fecha', 'pais', 'canal']).head(10)

Registros con combinación duplicada: 99


,fecha,pais,id_campaña,canal,gasto
105,2025-01-12,Argentina,organic_Argentina,NaN,1860.40
106,2025-01-12,Argentina,paid_search_Argentina,NaN,885.06
107,2025-01-12,Argentina,social_Argentina,NaN,2253.77
102,2025-01-12,Colombia,organic_Colombia,NaN,1819.27
103,2025-01-12,Colombia,paid_search_Colombia,NaN,1583.33
104,2025-01-12,Colombia,social_Colombia,NaN,712.64
99,2025-01-12,Mexico,organic_Mexico,NaN,2033.56
100,2025-01-12,Mexico,paid_search_Mexico,NaN,1260.65
101,2025-01-12,Mexico,social_Mexico,NaN,1660.90
114,2025-01-13,Argentina,organic_Argentina,NaN,833.13


In [ ]:
print('========== ORDERS ==========')
print(orders[['pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto']].describe())
print()

print('========== CATALOG ==========')
print(catalog[['nombre_producto','categoria_producto','proveedor']].describe())
print()

print('========== MARKETING =========')
print(marketing[['pais', 'id_campaña','canal']].describe())
print()

========== ORDERS ==========
            pais dispositivo fuente_referencia nombre_producto  \
count      24650       24926             24916           24916   
unique         6           2                 3               7   
top     Colombia     desktop            social  Blender-XL-Red   
freq        7467       12681              8385            4176   

       categoria_producto  
count               24916  
unique                  3  
top                 Hogar  
freq                 8346  

========== CATALOG ==========
        nombre_producto categoria_producto        proveedor
count                 7                  7                7
unique                7                  3                7
top     Phone-Pro-128GB        Electrónica  Mcmillan-Rhodes
freq                  1                  3                1

========== MARKETING =========
            pais             id_campaña        canal
count       1620                   1620         1519
unique         3               

In [ ]:
# Revisar variables categóricas en Orders
print("========== ORDERS ==========")
print("\n1. PAÍSES:")
print(orders['pais'].value_counts(dropna=False))

print("\n2. DISPOSITIVOS:")
print(orders['dispositivo'].value_counts(dropna=False))

print("\n3. FUENTE REFERENCIA:")
print(orders['fuente_referencia'].value_counts(dropna=False))

print("\n4. NOMBRE DE PRODUCTO:")
print(orders['nombre_producto'].value_counts(dropna=False))

print("\n5. CATEGORIA DE PRODUCTO:")
print(orders['categoria_producto'].value_counts(dropna=False))

========== ORDERS ==========

1. PAÍSES:
Colombia     7467
Mexico       7465
Argentina    7239
mexico        862
colombia      822
argentina     795
NaN           296
Name: pais, dtype: int64

2. DISPOSITIVOS:
desktop    12681
mobile     12245
NaN           20
Name: dispositivo, dtype: int64

3. FUENTE REFERENCIA:
social         8385
organic        8269
paid_search    8262
NaN              30
Name: fuente_referencia, dtype: int64

4. NOMBRE DE PRODUCTO:
Blender-XL-Red          4176
Vacuum-Pro-Black        4170
Jacket-Winter-M         4166
Sneakers-Urban-42       4129
Laptop-Gaming-16GB      2778
Tablet-Standard-64GB    2764
Phone-Pro-128GB         2733
NaN                       30
Name: nombre_producto, dtype: int64

5. CATEGORIA DE PRODUCTO:
Hogar          8346
Moda           8295
Electronica    8275
NaN              30
Name: categoria_producto, dtype: int64


In [ ]:
# Revisar variables categóricas en Catalog
print("========== CATALOG ==========")
print("\n1. NOMBRE DE PRODUCTO:")
print(catalog['nombre_producto'].value_counts(dropna=False))

print("\n2. CATEGORIA DE PRODUCTO:")
print(catalog['categoria_producto'].value_counts(dropna=False))

print("\n3. PROVEEDOR:")
print(catalog['proveedor'].value_counts(dropna=False))
print()

print('========== MARKETING ==========')
print("\n1. PAÍSES:")
print(marketing['pais'].value_counts(dropna=False))

print("\n2. ID CAMPAÑA:")
print(marketing['id_campaña'].value_counts(dropna=False))

print("\n3. CANAL:")
print(marketing['canal'].value_counts(dropna=False))

========== CATALOG ==========

1. NOMBRE DE PRODUCTO:
Phone-Pro-128GB         1
Laptop-Gaming-16GB      1
Tablet-Standard-64GB    1
Blender-XL-Red          1
Jacket-Winter-M         1
Sneakers-Urban-42       1
Vacuum-Pro-Black        1
Name: nombre_producto, dtype: int64

2. CATEGORIA DE PRODUCTO:
Electrónica    3
Hogar          2
Moda           2
Name: categoria_producto, dtype: int64

3. PROVEEDOR:
Mcmillan-Rhodes            1
King Ltd                   1
Long-Reid                  1
Fuller, Pena and Myers     1
Greene-Smith               1
Bowers LLC                 1
Rivera, Carr and Finley    1
Name: proveedor, dtype: int64

========== MARKETING ==========

1. PAÍSES:
Colombia     540
Mexico       540
Argentina    540
Name: pais, dtype: int64

2. ID CAMPAÑA:
paid_search_Argentina    180
social_Colombia          180
organic_Argentina        180
organic_Colombia         180
social_Mexico            180
paid_search_Mexico       180
paid_search_Colombia     180
organic_Mexico         

In [ ]:
# Estandarizar nombres
def estandarizar_nombres(columna):
    return columna.str.lower().str.strip().str.title()

orders['pais'] = estandarizar_nombres(orders['pais'])

# Estandarizar categoria en orders y catalog
orders['categoria_producto'] = orders['categoria_producto'].str.replace('Electronica', 'Electrónica')
catalog['categoria_producto'] = catalog['categoria_producto'].str.replace('Electronica', 'Electrónica')

# Validar cambios
print(orders['pais'].value_counts(dropna=False))
print()
print("Categorías en orders:")
print(orders['categoria_producto'].value_counts(dropna=False))

print("\nCategorías en catalog:")
print(catalog['categoria_producto'].value_counts(dropna=False))

Mexico       8327
Colombia     8289
Argentina    8034
NaN           296
Name: pais, dtype: int64

Categorías en orders:
Hogar          8346
Moda           8295
Electrónica    8275
NaN              30
Name: categoria_producto, dtype: int64

Categorías en catalog:
Electrónica    3
Hogar          2
Moda           2
Name: categoria_producto, dtype: int64


In [ ]:
# Valores faltantes en variables categóricas Orders
# Llenar Dispositivo con el valor más frecuente
orders['dispositivo'].fillna(orders['dispositivo'].mode()[0], inplace=True)
# Crear categoría especial
orders['pais'].fillna('Desconocido', inplace=True)

#Validar cambios
print("========== ORDERS ==========")
print("\n1. PAÍSES:")
print(orders['pais'].value_counts(dropna=False))

print("\n2. DISPOSITIVOS:")
print(orders['dispositivo'].value_counts(dropna=False))

print("\n3. FUENTE REFERENCIA:")
print(orders['fuente_referencia'].value_counts(dropna=False))

print("\n4. NOMBRE DE PRODUCTO:")
print(orders['nombre_producto'].value_counts(dropna=False))

print("\n5. CATEGORIA DE PRODUCTO:")
print(orders['categoria_producto'].value_counts(dropna=False))

========== ORDERS ==========

1. PAÍSES:
Mexico         8327
Colombia       8289
Argentina      8034
Desconocido     296
Name: pais, dtype: int64

2. DISPOSITIVOS:
desktop    12701
mobile     12245
Name: dispositivo, dtype: int64

3. FUENTE REFERENCIA:
social         8385
organic        8269
paid_search    8262
NaN              30
Name: fuente_referencia, dtype: int64

4. NOMBRE DE PRODUCTO:
Blender-XL-Red          4176
Vacuum-Pro-Black        4170
Jacket-Winter-M         4166
Sneakers-Urban-42       4129
Laptop-Gaming-16GB      2778
Tablet-Standard-64GB    2764
Phone-Pro-128GB         2733
NaN                       30
Name: nombre_producto, dtype: int64

5. CATEGORIA DE PRODUCTO:
Hogar          8346
Moda           8295
Electrónica    8275
NaN              30
Name: categoria_producto, dtype: int64


In [ ]:
# Analizar patrones de valores faltantes fuente_referencia
print("Registros con fuente_referencia faltante:")
registros_sin_fuente = orders[orders['fuente_referencia'].isnull()]
print(f"Total: {len(registros_sin_fuente)}")

# Ver si hay patrones por país, dispositivo o fecha
print("\nDistribución por país:")
print(registros_sin_fuente['pais'].value_counts())
print("\nDistribución por dispositivo:")
print(registros_sin_fuente['dispositivo'].value_counts())

Registros con fuente_referencia faltante:
Total: 30

Distribución por país:
Mexico       14
Argentina     9
Colombia      7
Name: pais, dtype: int64

Distribución por dispositivo:
mobile     15
desktop    15
Name: dispositivo, dtype: int64


In [ ]:
# Crear columna de mes
orders['mes'] = orders['fecha_hora_pedido'].dt.to_period('M')

# Calcular moda por país
moda_por_pais = (orders.dropna(subset=['fuente_referencia'])
                 .groupby('pais')['fuente_referencia']
                 .agg(lambda x: x.mode().iat[0]))

print("Moda por país:")
print(moda_por_pais)

# Función de imputación
def imputar_fuente(row):
    if pd.isna(row['fuente_referencia']):
        return moda_por_pais.get(row['pais'], 'organic')
    return row['fuente_referencia']

# Aplicar imputación
orders['fuente_referencia'] = orders.apply(imputar_fuente, axis=1)
print()
# Verificar que no quedan valores faltantes
print("Valores faltantes después de imputación:")
print(orders['fuente_referencia'].isnull().sum())

# Ver distribución final
print("\nDistribución final de fuente_referencia:")
print(orders['fuente_referencia'].value_counts())

Moda por país:
pais
Argentina          organic
Colombia            social
Desconocido    paid_search
Mexico              social
Name: fuente_referencia, dtype: object

Valores faltantes después de imputación:
0

Distribución final de fuente_referencia:
social         8406
organic        8278
paid_search    8262
Name: fuente_referencia, dtype: int64


In [ ]:
# Analizar registros con nombre_producto faltante
print("Registros con nombre_producto faltante:")
registros_sin_producto = orders[orders['nombre_producto'].isnull()]
print(f"Total: {len(registros_sin_producto)}")

# Ver si hay patrones
print("\nDistribución por país:")
print(registros_sin_producto['pais'].value_counts())

print("\nDistribución por categoría:")
print(registros_sin_producto['categoria_producto'].value_counts())

print("\nDistribución por fuente:")
print(registros_sin_producto['fuente_referencia'].value_counts())

Registros con nombre_producto faltante:
Total: 30

Distribución por país:
Mexico       14
Argentina     9
Colombia      7
Name: pais, dtype: int64

Distribución por categoría:
Series([], Name: categoria_producto, dtype: int64)

Distribución por fuente:
social     21
organic     9
Name: fuente_referencia, dtype: int64


In [ ]:
# Eliminar registros con nombre_producto faltante
orders_clean = orders.dropna(subset=['nombre_producto'])

# Verificar el resultado
print(f"Registros originales: {len(orders)}")
print(f"Registros después de eliminar faltantes: {len(orders_clean)}")
print(f"Registros eliminados: {len(orders) - len(orders_clean)}")

# Actualizar el dataset principal
orders = orders_clean.copy().reset_index(drop=True)

# Verificar que no quedan valores faltantes
print("Valores faltantes en nombre_producto:")
print(orders['nombre_producto'].isnull().sum())

print("\n NOMBRE DE PRODUCTO:")
print(orders['nombre_producto'].value_counts(dropna=False))

print("\n CATEGORIA DE PRODUCTO:")
print(orders['categoria_producto'].value_counts(dropna=False))

Registros originales: 24946
Registros después de eliminar faltantes: 24916
Registros eliminados: 30
Valores faltantes en nombre_producto:
0

 NOMBRE DE PRODUCTO:
Blender-XL-Red          4176
Vacuum-Pro-Black        4170
Jacket-Winter-M         4166
Sneakers-Urban-42       4129
Laptop-Gaming-16GB      2778
Tablet-Standard-64GB    2764
Phone-Pro-128GB         2733
Name: nombre_producto, dtype: int64

 CATEGORIA DE PRODUCTO:
Hogar          8346
Moda           8295
Electrónica    8275
Name: categoria_producto, dtype: int64


In [ ]:
# Analizar registros con categoria_producto faltante
print("Registros con categoria_producto faltante:")
registros_sin_categoria = orders[orders['categoria_producto'].isnull()]
print(f"Total: {len(registros_sin_categoria)}")

# Ver qué productos tienen categoría faltante
print("\nProductos sin categoría:")
print(registros_sin_categoria['nombre_producto'].value_counts())

# Ver si podemos usar el catálogo para completar
print("\nCategorías disponibles en el catálogo:")
print(catalog[['nombre_producto', 'categoria_producto']])

Registros con categoria_producto faltante:
Total: 0

Productos sin categoría:
Series([], Name: nombre_producto, dtype: int64)

Categorías disponibles en el catálogo:
        nombre_producto categoria_producto
0    Laptop-Gaming-16GB        Electrónica
1       Phone-Pro-128GB        Electrónica
2  Tablet-Standard-64GB        Electrónica
3        Blender-XL-Red              Hogar
4      Vacuum-Pro-Black              Hogar
5     Sneakers-Urban-42               Moda
6       Jacket-Winter-M               Moda


In [ ]:
# Crear diccionario de mapeo desde el catálogo
mapeo_categorias = dict(zip(catalog['nombre_producto'], catalog['categoria_producto']))
print("Mapeo de categorías desde el catálogo:")
print(mapeo_categorias)

# Función para imputar categorías faltantes
def imputar_categoria(row):
    if pd.isna(row['categoria_producto']):
        # Buscar la categoría en el catálogo usando el nombre del producto
        return mapeo_categorias.get(row['nombre_producto'], row['categoria_producto'])
    return row['categoria_producto']

# Aplicar la función
orders['categoria_producto'] = orders.apply(imputar_categoria, axis=1)
print()
# Verificar que no quedan valores faltantes
print("Valores faltantes en categoria_producto después de imputación:")
print(orders['categoria_producto'].isnull().sum())

# Ver distribución final
print("\nDistribución final de categorías:")
print(orders['categoria_producto'].value_counts())

# Verificar algunos ejemplos de registros que fueron imputados
print("\nEjemplos de registros que tenían categoría faltante:")
print(orders[orders['nombre_producto'].isin(['Sneakers-Urban-42', 'Jacket-Winter-M'])][
    ['nombre_producto', 'categoria_producto']].head(10))

Mapeo de categorías desde el catálogo:
{'Laptop-Gaming-16GB': 'Electrónica', 'Phone-Pro-128GB': 'Electrónica', 'Tablet-Standard-64GB': 'Electrónica', 'Blender-XL-Red': 'Hogar', 'Vacuum-Pro-Black': 'Hogar', 'Sneakers-Urban-42': 'Moda', 'Jacket-Winter-M': 'Moda'}

Valores faltantes en categoria_producto después de imputación:
0

Distribución final de categorías:
Hogar          8346
Moda           8295
Electrónica    8275
Name: categoria_producto, dtype: int64

Ejemplos de registros que tenían categoría faltante:
      nombre_producto categoria_producto
0     Jacket-Winter-M               Moda
9   Sneakers-Urban-42               Moda
12    Jacket-Winter-M               Moda
13    Jacket-Winter-M               Moda
15    Jacket-Winter-M               Moda
17  Sneakers-Urban-42               Moda
19    Jacket-Winter-M               Moda
23    Jacket-Winter-M               Moda
24  Sneakers-Urban-42               Moda
34  Sneakers-Urban-42               Moda


In [ ]:
# Analizar registros con canal faltante para Marketing
print("Registros con canal faltante:")
registros_sin_canal = marketing[marketing['canal'].isnull()]
print(f"Total: {len(registros_sin_canal)}")

# Ver qué campaña tienen canal faltante
print("\nCampaña sin canal:")
print(registros_sin_canal['id_campaña'].value_counts())

# Ver si podemos usar id_campaña para completar
print("\nCanales disponibles en id_campaña:")
print(marketing[['id_campaña', 'canal']])

Registros con canal faltante:
Total: 101

Campaña sin canal:
organic_Mexico           12
social_Argentina         12
paid_search_Argentina    11
paid_search_Colombia     11
organic_Argentina        11
organic_Colombia         11
paid_search_Mexico       11
social_Colombia          11
social_Mexico            11
Name: id_campaña, dtype: int64

Canales disponibles en id_campaña:
                 id_campaña        canal
0            organic_Mexico      organic
1        paid_search_Mexico  paid_search
2             social_Mexico       social
3          organic_Colombia      organic
4      paid_search_Colombia  paid_search
...                     ...          ...
1615   paid_search_Colombia  paid_search
1616        social_Colombia       social
1617      organic_Argentina      organic
1618  paid_search_Argentina  paid_search
1619       social_Argentina       social

[1620 rows x 2 columns]


In [ ]:
# Función para extraer canal desde id_campaña
def extraer_canal_desde_id(id_campaña):
    if pd.isna(id_campaña):
        return None

    # Si empieza con "paid_search", devolver "paid_search"
    if 'paid_search' in id_campaña:
        return 'paid_search'
    # Para otros casos, tomar la primera parte
    return id_campaña.split('_')[0]

# Aplicar solo a registros con canal faltante
mask_canal_faltante = marketing['canal'].isna()
marketing.loc[mask_canal_faltante, 'canal'] = marketing.loc[mask_canal_faltante, 'id_campaña'].apply(extraer_canal_desde_id)

# Verificar que no quedan valores faltantes
print("Valores faltantes en canal después de imputación:")
print(marketing['canal'].isnull().sum())

# Verificar algunos ejemplos de registros que fueron imputados
print("\nEjemplos de registros que tenían canal faltante:")
print(marketing[marketing['id_campaña'].isin(['organic_Mexico', 'social_Argentina', 'paid_search_Argentina'])][
    ['id_campaña', 'canal']].head(10))

print("\n CANAL:")
print(marketing['canal'].value_counts(dropna=False))

# Convertir formatos
orders['cantidad'] = orders['cantidad'].astype('int64')
orders['monto_descuento'] = orders['monto_descuento'].astype('int64')

orders.info()

Valores faltantes en canal después de imputación:
0

Ejemplos de registros que tenían canal faltante:
               id_campaña        canal
0          organic_Mexico      organic
7   paid_search_Argentina  paid_search
8        social_Argentina       social
9          organic_Mexico      organic
16  paid_search_Argentina  paid_search
17       social_Argentina       social
18         organic_Mexico      organic
25  paid_search_Argentina  paid_search
26       social_Argentina       social
27         organic_Mexico      organic

 CANAL:
organic        540
social         540
paid_search    540
Name: canal, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24916 entries, 0 to 24915
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           24916 non-null  object        
 1   id_usuario          24916 non-null  object        
 2   fecha_hora_pedido   24916 non-null  dat

**Documentación**

**Variables númericas con negativos y ceros inválidos:** Se encontraron 4 registros con valores negativos en el dataset orders en las columnas `cantidad` y `monto_total`, se decide eliminar estos registros, ya que:
- Representan errores de captura de datos, no tienen sentido lógico en el contexto los valores negativos.
- Son pocos registros por  lo que su eliminación no afectará el análisis.
- Mantienen la integridad del análisis y conservar su valor distorsionaría cálculos.

**Valores nulos:** Se encontraron 50 registros nulos que comparten las columnas `cantidad`, `presio_unitario`, `monto_descuento`, despues de an alizar se decide eliminar estos registros, ya que, son críticos para el análisis, suimpacto es mínimo y esto mantiene la calidad.

**Consistencia de montos:** Se detectaron 1.146 registros con diferencias monetarias, las diferencias son exactamente 0.01. Por lo tanto, no afecta significativamente el análsiis de rentabilidad, representa errores de redondeo técnicos. Se usará el registro original `monto_total`

**Duplicados:**
- Se encontraron 100 duplicados en orders, se realizo la busqueda por la columna `id_pedido` ya que esta debe tener registros únicos. Se decide eliminar los duplicados y dejar el primer registro para no inflar cálculos ni afectar el análsis.
- Se encontraron 66 duplicados en marketing, por una combinación clave (`fecha`, `país`,`canal`) pero después de analizar las columnas se observa que es normal, son registros y campañas legítimas, así que, se mantienen.

**Variables categóricas:**
- Se encuentra inconsistencias en los nombres de los paises en el dataset Orders como Colombia (7,481), colombia (823), Mexico (7,478), mexico (863), Argentina (7,259), argentina (796), se estandariza para que cuente en el mismo país.
- Tambien se encuentran inconsistencias en categorias del producto en el dataset orders: Electronica (8,275) y en catalog: Electrónica (3) - con tilde. Se remplaza para que queden con tilde y no tener problemas para el merge entre los datasets.

**Variables categóricas con valores faltantes:**

Se encontró que en Orders:
- Para `dispositivo` 20 valores NaN de 24,996 registros (0.08%): Se imputa con la moda ya que son muy pocos casos.
- Para `pais` 296 valores NaN de 24,996 registros (1.2%): Se crea categoría "Desconocido" porque: representa información valiosa (usuarios sin geolocalización), puede ser útil para análisis de marketing y no distorsiona las proporciones reales por país
- En `fuente_referencia` se encontraron 30 NaN, por lo que se analiza el patrón y se decide realizar una imputación por moda estratificada, ya que, preserva patrones reales, su impacto es mínimo y no sesgará métricas y se evita crear una categoría artificial que podría confundir insights.
- En `nombre_producto` se encontraron 30 valores faltantes, se observa que estos registros tambien tienen `categoria_prodcto` como NaN, así que, se decide eliminar los registros, ya que son solo 0.12% de los datos un impacto bajo, se evitan sesgos al introducir información artificial y se mantiene la calidad.
- En `categoria_producto` se encuentran aún 50 registros con valores faltantes, despues del paso anterior, se analizan los registros y se decide usar `nombre_producto` del dataset Catalog para imputar su respectiva categoria.

Para Marketing:
- Se encontraron 101 valores faltantes en la columna `canal`, se analizan los registros y se decide usar `id_campaña` para imputar su respectivo canal.

In [ ]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False, float_format='%.2f')
catalog.to_csv('catalog_clean.csv', index=False, float_format='%.2f')
marketing.to_csv('marketing_clean.csv', index=False, float_format='%.2f')

---

## 🔹 Paso 2: Analizar si el negocio es rentable

In [ ]:
# Calcular ingreso total
ingreso_total = orders['monto_total'].sum()

# Calcular costo total
# Combinar orders con catalog para obtener el costo unitario
orders_catalog = orders.merge(catalog[['nombre_producto', 'costo_unitario']],
                                 on='nombre_producto',
                                 how='left')

# Calcular el costo total por pedido
orders_catalog['costo_total_pedido'] = orders_catalog['cantidad'] * orders_catalog['costo_unitario']

# Sumar todos los costos
costo_total = orders_catalog['costo_total_pedido'].sum()

# Calcular inversión en marketing
inversion_total_marketing = marketing['gasto'].sum()

print(f'Ingreso total: {ingreso_total:.2f}')
print(f'Costo total: {costo_total:.2f}')
print('Inversión en Marketing:', inversion_total_marketing)

Ingreso total: 51954718.94
Costo total: 43124069.01
Inversión en Marketing: 2871843.53


In [ ]:
profit = ingreso_total - costo_total - inversion_total_marketing
print(f'Profit: {profit:.2f}')

Profit: 5958806.40


In [ ]:

ticket_promedio = orders['monto_total'].mean()
cantidad_promedio = orders['cantidad'].mean()
producto_mas_vendido = orders.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False)
gasto_marketing_por_canal = marketing.groupby('canal')['gasto'].sum().sort_values(ascending=False)

print(f'Ticket Promedio: {ticket_promedio:.2f}')
print(f'Cantidad promedio de productos por orden:  {cantidad_promedio:.2f}')
print('\nProducto más vendido:\n', producto_mas_vendido)
print('\nGasto de Marketing por Canal:\n', gasto_marketing_por_canal)

Ticket Promedio: 2085.20
Cantidad promedio de productos por orden:  7.12

Producto más vendido:
 nombre_producto
Laptop-Gaming-16GB      144198
Vacuum-Pro-Black          6284
Blender-XL-Red            6279
Jacket-Winter-M           6256
Sneakers-Urban-42         6172
Tablet-Standard-64GB      4153
Phone-Pro-128GB           4140
Name: cantidad, dtype: int64

Gasto de Marketing por Canal:
 canal
social         976818.37
organic        972650.96
paid_search    922374.20
Name: gasto, dtype: float64


**Rentabilidad del negocio**

- El ingreso total es de: 51'954,718.94
- El costo total es de: 43'124,069.01
- La inversión total en marketing es de: 2'871,843.53
- Tiene un profit de: 5'958,806.40 lo cual muestra que el negocio es rentable

**Comportamiento de ventas**
- El ticket promedio por orden es de: 2,085.20
- La cantidad promedio de productos por orden es de: 7.12
- El producto más vendido es Laptop-Gaming-16GB con 144,198 unidades, seguido de Vacuum-Pro-Black con 6,284 unidades.

Se ha gastado en marketing por canal:
- social: 976,818.37
- organic: 972,650.96
- paid_search: 922,374.20

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.


In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [ ]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) as usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY usuarios_unicos DESC;
'''
totals = pd.read_sql(query_totals, con=engine)
totals

In [ ]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH funnel_data AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) as usuarios_unicos
    FROM events
    GROUP BY nombre_evento
),
funnel_ordered AS (
    SELECT
        nombre_evento,
        usuarios_unicos,
        CASE
            WHEN nombre_evento = 'first_visit' THEN 1
            WHEN nombre_evento = 'add_to_cart' THEN 2
            WHEN nombre_evento = 'select_item' THEN 3
            WHEN nombre_evento = 'begin_checkout' THEN 4
            WHEN nombre_evento = 'add_payment_info' THEN 5
            WHEN nombre_evento = 'purchase' THEN 6
        END as orden_paso
    FROM funnel_data
)
SELECT
    nombre_evento,
    usuarios_unicos,
    LAG(usuarios_unicos) OVER (ORDER BY orden_paso) as usuarios_paso_anterior,
    ROUND(
        (usuarios_unicos * 100.0 / LAG(usuarios_unicos) OVER (ORDER BY orden_paso)), 2
    ) as tasa_conversion_pct
FROM funnel_ordered
ORDER BY orden_paso;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)


In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

In [ ]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT
        id_usuario,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) as cohorte_mes,
        fecha_registro
    FROM users
),
actividad_semanal AS (
    SELECT
        c.id_usuario,
        c.cohorte_mes,
        ua.dias_despues_registro,
        ua.activo,
        CASE
            WHEN ua.dias_despues_registro BETWEEN 1 AND 7 THEN 'retenido_w1'
            WHEN ua.dias_despues_registro BETWEEN 8 AND 14 THEN 'retenido_w2'
            WHEN ua.dias_despues_registro BETWEEN 15 AND 21 THEN 'retenido_w3'
        END as periodo_semana
    FROM cohortes c
    JOIN user_activity ua ON c.id_usuario = ua.id_usuario
    WHERE ua.dias_despues_registro <= 21
        AND ua.activo = 1
),
usuarios_retenidos AS (
    SELECT
        cohorte_mes,
        periodo_semana,
        COUNT(DISTINCT id_usuario) as usuarios_activos
    FROM actividad_semanal
    WHERE periodo_semana IS NOT NULL
    GROUP BY cohorte_mes, periodo_semana
),
tamaño_cohortes AS (
    SELECT
        cohorte_mes,
        COUNT(DISTINCT id_usuario) as usuarios_iniciales
    FROM cohortes
    GROUP BY cohorte_mes
)
SELECT
    tc.cohorte_mes,
    tc.usuarios_iniciales,

    ROUND(
        100.0 * MAX(CASE WHEN ur.periodo_semana = 'retenido_w1'
            THEN ur.usuarios_activos END)
        / tc.usuarios_iniciales,
        2
    ) AS semana_1,

    ROUND(
        100.0 * MAX(CASE WHEN ur.periodo_semana = 'retenido_w2'
            THEN ur.usuarios_activos END)
        / tc.usuarios_iniciales,
        2
    ) AS semana_2,

    ROUND(
        100.0 * MAX(CASE WHEN ur.periodo_semana = 'retenido_w3'
            THEN ur.usuarios_activos END)
        / tc.usuarios_iniciales,
        2
    ) AS semana_3

FROM tamaño_cohortes tc
LEFT JOIN usuarios_retenidos ur
    ON tc.cohorte_mes = ur.cohorte_mes

GROUP BY tc.cohorte_mes, tc.usuarios_iniciales
ORDER BY tc.cohorte_mes;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

**Insights**
- La mayor pérdida de usuarios se produce entre `begin_checkout` y `add_payment_info` hay una caída del 13.29% (86.71% de conversión), esta es la etapa más crítica del funnel.
- La conversión de `add_payment_info` a `purchase` es excelente (99.84%), los usuarios que llegan a agregar información de pago casi siempre completan la compra.
- Del total de visitantes, 80.04% completan la compra (6,240/7,796).
- Febrero es parece especial, ya que, es la única cohorte que muestra crecimiento sostenido hasta la semana 3 (43.98%)
- Retención general estable: La mayoría de cohortes mantienen ~41-43% de retención
- Como patrón interesante, varias cohortes muestran recuperación en semana 3

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)


---
Hipótesis estadística
   - **H₀ (Hipótesis nula):**  La tasa de conversión es igual en ambas variantes. Cualquier diferencia se debe al azar del muestreo.
   - **H₁ (Hipótesis alternativa):** a tasa de conversión es diferente entre ambas variantes. Hay un factor que influye en la decisión de compra.
   
**Test estadístico:** z-test

**Nivel de significancia alpha:** α = 0.05 (5%)


In [ ]:
from statsmodels.stats.proportion import proportions_ztest
import pandas as pd
experiment = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')

In [ ]:
experiment.info()
experiment.head()

In [ ]:
# Analizar distribución por grupo
print("Distribución por variante:")
print(experiment['variante'].value_counts())
print()

# Calcular tasas de conversión por grupo
conversion_by_group = experiment.groupby('variante')['convirtio'].agg(['count', 'sum', 'mean'])
conversion_by_group.columns = ['n_usuarios', 'conversiones', 'tasa_conversion']
print("Tasas de conversión por grupo:")
print(conversion_by_group)

In [ ]:
# Ejecutar z-test para proporciones
conversiones = [779, 820]  # conversiones por grupo
n_usuarios = [4965, 5035]  # usuarios por grupo

z_stat, p_value = proportions_ztest(conversiones, n_usuarios) #z-test

# Mostrar resultados
print("========== RESULTADOS DEL Z-TEST ==========")
print(f"z_stat: {z_stat:.4f}")
print(f"p_value: {p_value:.4f}")
print()

# Interpretación
if p_value < 0.05:
    print('Rechazamos la hipótesis nula: hay evidencia de una diferencia.')
else:
    print('No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia.')

**Conclusión e interpretación:**
- Decisión: No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia.

El resultado no es estadísticamente significativo de que el cambio en el checkout UI haya impactado significativamente la conversión. La diferencia observada (15.69% vs 16.29%) puede explicarse por variabilidad natural del muestreo, no por el tratamiento

- Recomendación de negocio: No implementar el cambio, ya que no demostró un impacto estadísticamente significativo. Explorar algún análisis adicional, por segmentación para ver si hay diferencias en subgrupos específicos.

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

---